In [9]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for headless execution
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


def load_ts_data(filepath):
    """Load time series data from .txt file (UCR/UEA format)"""
    data = []
    labels = []
    with open(filepath, 'r') as f:
        for line in f:
            values = line.strip().split()
            if len(values) > 1:
                labels.append(int(float(values[0])))
                data.append([float(v) for v in values[1:]])
    return np.array(data), np.array(labels)


print("=" * 60)
print("TIME SERIES CLASSIFICATION - SYMBOLS DATASET")
print("=" * 60)


TIME SERIES CLASSIFICATION - SYMBOLS DATASET


In [10]:
# =============================================================================
# ЗАДАНИЕ 1: Выбор и обоснование набора данных
# ================================================================================
# Набор данных: Symbols (из архива временных рядов UCR)
#
# Обоснование:
# - Набор данных Symbols является известным эталонным набором данных для классификации временных рядов из архива UCR.
# # - Содержит 6 классов, представляющих различные формы символов (Треугольник, Круг,
# Труба, Колокол, Двойной колокол, Спираль).
# - Данные уже нормализованы (среднее значение = 0, стандартное отклонение = 1), что делает их подходящими для
# различных подходов к классификации без обширной предварительной обработки.
# - С 25 обучающими и 995 тестовыми выборками он предоставляет хорошую тестовую площадку
# для оценки различных методов классификации.
# - 398 временных точек на серию позволяют извлекать значимые закономерности.
# ==============================================================================
# 1. Загрузка набора данных

print("\n1. Loading Symbols dataset...")
X_train, y_train = load_ts_data('Symbols/Symbols_TRAIN.txt')
X_test, y_test = load_ts_data('Symbols/Symbols_TEST.txt')

# Reshape for aeon (n_cases, n_channels, n_timepoints)
X_train = X_train.reshape(-1, 1, X_train.shape[1])
X_test = X_test.reshape(-1, 1, X_test.shape[1])

print(f"   Training samples: {X_train.shape[0]}")
print(f"   Test samples: {X_test.shape[0]}")
print(f"   Time series length: {X_train.shape[2]}")
print(f"   Number of classes: {len(np.unique(y_train))}")
print(f"   Classes: {np.unique(y_train)}")

# 2. Dataset Analysis
print("\n2. Dataset Analysis:")
print("-" * 40)

# Class distribution
for cls in np.unique(y_train):
    count = np.sum(y_train == cls)
    print(f"   Class {int(cls)}: {count} training samples ({100*count/len(y_train):.1f}%)")

# Time series statistics
print(f"\n   Time series length: {X_train.shape[2]}")
print(f"   Value range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"   Mean value: {X_train.mean():.3f}")
print(f"   Std deviation: {X_train.std():.3f}")


1. Loading Symbols dataset...
   Training samples: 25
   Test samples: 995
   Time series length: 398
   Number of classes: 6
   Classes: [1 2 3 4 5 6]

2. Dataset Analysis:
----------------------------------------
   Class 1: 8 training samples (32.0%)
   Class 2: 5 training samples (20.0%)
   Class 3: 3 training samples (12.0%)
   Class 4: 3 training samples (12.0%)
   Class 5: 3 training samples (12.0%)
   Class 6: 3 training samples (12.0%)

   Time series length: 398
   Value range: [-2.311, 2.205]
   Mean value: -0.000
   Std deviation: 0.999


In [11]:
# =============================================================================
# ЗАДАНИЕ 2: Визуализация данных
# =============================================================================
print("\n3. Visualizing data...")

# Map class labels to symbol names
CLASS_NAMES = {1: 'Triangle', 2: 'Circle', 3: 'Trumpet', 4: 'Bell', 5: 'Double Bell', 6: 'Spiral'}

# Plot sample time series for each class
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, cls in enumerate(np.unique(y_train)):
    class_samples = X_train[y_train == cls]
    sample = class_samples[0].flatten()
    axes[idx].plot(sample, linewidth=1.5)
    axes[idx].set_title(f'Class {cls}: {CLASS_NAMES[cls]}')
    axes[idx].set_xlabel('Timepoint')
    axes[idx].set_ylabel('Value')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('symbols_samples.png', dpi=100, bbox_inches='tight')
plt.close()

# Отобразите все обучающие выборки, раскрашенные по классам.
plt.figure(figsize=(15, 6))
for cls in np.unique(y_train):
    class_samples = X_train[y_train == cls]
    for sample in class_samples:
        plt.plot(sample.flatten(), alpha=0.5, label=f'Class {cls}' if sample is class_samples[0] else '')
plt.title('All Training Time Series Colored by Class')
plt.xlabel('Timepoint')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('symbols_all_series.png', dpi=100, bbox_inches='tight')
plt.close()

print("   Data visualization complete. (saved to symbols_samples.png, symbols_all_series.png)")



3. Visualizing data...
   Data visualization complete. (saved to symbols_samples.png, symbols_all_series.png)


In [15]:
# ============================================================================
# ЗАДАНИЕ 3: Методы классификации временных рядов
# ============================================================================
print("\n" + "=" * 60)
print("TASK 3: TIME SERIES CLASSIFICATION METHODS")
print("=" * 60)

# -----------------------------------------------------------------------------
# 3.1 Методы на основе расстояний
# -----------------------------------------------------------------------------
print("\n" + "-" * 60)
print("3.1 DISTANCE-BASED METHODS")
print("-" * 60)

from aeon.classification.distance_based import (
    KNeighborsTimeSeriesClassifier,
    ElasticEnsemble,
    ProximityForest
)
from aeon.distances import dtw_distance, adtw_distance, msm_distance

# Модель 1: KNN с расстоянием DTW
print("\n[Model 1] KNN with DTW distance...")
knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=3, distance=dtw_distance)
knn_dtw.fit(X_train, y_train)
preds_dtw = knn_dtw.predict(X_test)
acc_dtw = accuracy_score(y_test, preds_dtw)
print(f"   Accuracy: {acc_dtw:.4f}")

# Модель 2: KNN с расстоянием ADTW
print("\n[Model 2] KNN with ADTW distance...")
knn_adtw = KNeighborsTimeSeriesClassifier(n_neighbors=3, distance=adtw_distance)
knn_adtw.fit(X_train, y_train)
preds_adtw = knn_adtw.predict(X_test)
acc_adtw = accuracy_score(y_test, preds_adtw)
print(f"   Accuracy: {acc_adtw:.4f}")

# Модель 3: KNN с расстоянием MSM
print("\n[Model 3] KNN with MSM distance...")
knn_msm = KNeighborsTimeSeriesClassifier(n_neighbors=3, distance=msm_distance)
knn_msm.fit(X_train, y_train)
preds_msm = knn_msm.predict(X_test)
acc_msm = accuracy_score(y_test, preds_msm)
print(f"   Accuracy: {acc_msm:.4f}")

# Модель 4: Elastic Ensemble (пропускаем для маленьких датасетов)
print("\n[Model 4] Elastic Ensemble... (skipped - requires larger dataset)")
acc_ee = 0.0
print("   Skipped - requires larger training set")

# Итоговая сводка по методам на основе расстояний
print("\n>>> Distance-based Methods Summary:")
print(f"   KNN-DTW:     {acc_dtw:.4f}")
print(f"   KNN-ADTW:    {acc_adtw:.4f}")
print(f"   KNN-MSM:     {acc_msm:.4f}")
print(f"   ElasticEns:  {acc_ee:.4f}")


TASK 3: TIME SERIES CLASSIFICATION METHODS

------------------------------------------------------------
3.1 DISTANCE-BASED METHODS
------------------------------------------------------------

[Model 1] KNN with DTW distance...
   Accuracy: 0.9497

[Model 2] KNN with ADTW distance...
   Accuracy: 0.9286

[Model 3] KNN with MSM distance...
   Accuracy: 0.9317

[Model 4] Elastic Ensemble... (skipped - requires larger dataset)
   Skipped - requires larger training set

>>> Distance-based Methods Summary:
   KNN-DTW:     0.9497
   KNN-ADTW:    0.9286
   KNN-MSM:     0.9317
   ElasticEns:  0.0000


In [16]:
# -----------------------------------------------------------------------------
# 3.2 Методы на основе сверток (Rocket, Hydra)
# -----------------------------------------------------------------------------
print("\n" + "-" * 60)
print("3.2 CONVOLUTION-BASED METHODS (Rocket, Hydra)")
print("-" * 60)

from aeon.classification.convolution_based import (
    RocketClassifier,
    MiniRocketClassifier,
    MultiRocketClassifier,
    HydraClassifier,
    MultiRocketHydraClassifier
)

# Модель 1: Классификатор Rocket
print("\n[Model 1] Rocket Classifier...")
rocket = RocketClassifier(n_jobs=-1, random_state=42, n_kernels=100)
rocket.fit(X_train, y_train)
preds_rocket = rocket.predict(X_test)
acc_rocket = accuracy_score(y_test, preds_rocket)
print(f"   Accuracy: {acc_rocket:.4f}")

# Модель 2: Классификатор MiniRocket
print("\n[Model 2] MiniRocket Classifier...")
minirocket = MiniRocketClassifier(n_jobs=-1, random_state=42, n_kernels=100)
minirocket.fit(X_train, y_train)
preds_minirocket = minirocket.predict(X_test)
acc_minirocket = accuracy_score(y_test, preds_minirocket)
print(f"   Accuracy: {acc_minirocket:.4f}")

# Модель 3: Классификатор MultiRocket
print("\n[Model 3] MultiRocket Classifier...")
multirocket = MultiRocketClassifier(n_jobs=-1, random_state=42, n_kernels=100)
multirocket.fit(X_train, y_train)
preds_multirocket = multirocket.predict(X_test)
acc_multirocket = accuracy_score(y_test, preds_multirocket)
print(f"   Accuracy: {acc_multirocket:.4f}")

# Модель 4: Классификатор Hydra
print("\n[Model 4] Hydra Classifier...")
hydra = HydraClassifier(n_jobs=-1, random_state=42)
hydra.fit(X_train, y_train)
preds_hydra = hydra.predict(X_test)
acc_hydra = accuracy_score(y_test, preds_hydra)
print(f"   Accuracy: {acc_hydra:.4f}")

# Модель 5: Комбинированный классификатор MultiRocket + Hydra
print("\n[Model 5] MultiRocket + Hydra Classifier...")
mr_hydra = MultiRocketHydraClassifier(n_jobs=-1, random_state=42, n_kernels=100)
mr_hydra.fit(X_train, y_train)
preds_mr_hydra = mr_hydra.predict(X_test)
acc_mr_hydra = accuracy_score(y_test, preds_mr_hydra)
print(f"   Accuracy: {acc_mr_hydra:.4f}")

# Итоговая сводка по методам на основе сверток
print("\n>>> Convolution-based Methods Summary:")
print(f"   Rocket:        {acc_rocket:.4f}")
print(f"   MiniRocket:    {acc_minirocket:.4f}")
print(f"   MultiRocket:   {acc_multirocket:.4f}")
print(f"   Hydra:         {acc_hydra:.4f}")
print(f"   MR+Hydra:      {acc_mr_hydra:.4f}")


------------------------------------------------------------
3.2 CONVOLUTION-BASED METHODS (Rocket, Hydra)
------------------------------------------------------------

[Model 1] Rocket Classifier...
   Accuracy: 0.9729

[Model 2] MiniRocket Classifier...
   Accuracy: 0.9698

[Model 3] MultiRocket Classifier...
   Accuracy: 0.9779

[Model 4] Hydra Classifier...
   Accuracy: 0.9789

[Model 5] MultiRocket + Hydra Classifier...
   Accuracy: 0.9799

>>> Convolution-based Methods Summary:
   Rocket:        0.9729
   MiniRocket:    0.9698
   MultiRocket:   0.9779
   Hydra:         0.9789
   MR+Hydra:      0.9799
